# Truth by Design at Scale - Final Three-Agent MoE Benchmark

This is the final reproducible notebook for Objectives 1-3 in the project proposal.

## Experimental contract

- No web search or retrieval tools are used.
- Gold labels from the final evaluation set are never included in model prompts.
- Data are split into three disjoint sets:
  1. `prompt_development`: supplies few-shot label-boundary examples only;
  2. `calibration`: estimates model reliability and routing weights only;
  3. `evaluation`: produces the reported final metrics only.
- Each expert evaluates the logical sequence Pauli-Frankfurt -> CRAAP -> Ethics -> final verdict in one structured API call. This preserves gate order while reducing API calls.
- The proposal strategies are implemented: majority voting, confidence-weighted voting, and gate-specific routing.
- Calibration-accuracy weighting is included as an additional exploratory strategy.
- The inherited `Pred_Verdict` column is reported as an inherited prior-study baseline, not a reproduced GPT baseline.

Start with the pilot. Set `PILOT_PER_DATASET = None` only after the pilot passes all quality checks.


In [ ]:
from __future__ import annotations

import hashlib, json, math, os, random, re, time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, Iterable, Optional

import numpy as np
import pandas as pd

ROOT = Path.cwd()
DATA_DIR = ROOT / "Dataset"
OUTPUT_DIR = ROOT / "outputs" / "final_moe_benchmark"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_JSONL = OUTPUT_DIR / "model_outputs.jsonl"
ENSEMBLE_CSV = OUTPUT_DIR / "ensemble_predictions.csv"
METRICS_CSV = OUTPUT_DIR / "benchmark_metrics.csv"
MCNEMAR_CSV = OUTPUT_DIR / "mcnemar_tests.csv"
DISAGREEMENT_CSV = OUTPUT_DIR / "disagreement_analysis.csv"
MANIFEST_JSON = OUTPUT_DIR / "dataset_manifest.json"
SUMMARY_JSON = OUTPUT_DIR / "run_summary.json"

SEED = 20260720
RUN_LIVE_APIS = True
PILOT_PER_DATASET = None       # None = all evaluation claims
PROMPT_EXAMPLES_PER_CLASS = 2
CALIBRATION_PER_CLASS = 8
MAX_RETRIES = 4
RETRY_BASE_SECONDS = 4
MAX_PARALLEL_PROVIDERS = 3

OPENAI_MODEL = "gpt-5.4"
CLAUDE_MODEL = "claude-sonnet-5"
GEMINI_MODEL = "gemini-2.5-pro"

LABELS = ["Pants on Fire", "False", "Mostly False", "Half True", "Mostly True", "True"]
LABEL_TO_SCORE = dict(zip(LABELS, np.linspace(0.0, 1.0, len(LABELS))))
RISK_TO_SCORE = {"low": 0.0, "medium": 0.5, "high": 1.0}

print("Output directory:", OUTPUT_DIR)
print("Live API calls:", RUN_LIVE_APIS)
print("Pilot per dataset:", PILOT_PER_DATASET)
print("Models:", OPENAI_MODEL, CLAUDE_MODEL, GEMINI_MODEL)


In [ ]:
def load_env_file(path: Path) -> None:
    if not path.exists():
        return
    for raw in path.read_text(encoding="utf-8").splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))

load_env_file(ROOT / "api_keys.env")

ALIASES = {
    "pants on fire!": "Pants on Fire", "pants-on-fire": "Pants on Fire",
    "partially true": "Half True", "partially_true": "Half True",
    "mostly-false": "Mostly False", "half-true": "Half True", "mostly-true": "Mostly True",
}

def clean_claim(value: Any) -> str:
    if pd.isna(value): return ""
    return re.sub(r"\s+", " ", str(value).strip().strip('"').strip("'"))

def normalize_label(value: Any) -> str:
    if pd.isna(value): return ""
    text = re.sub(r"[_\s]+", " ", str(value).strip()).strip()
    key = text.lower()
    if key in ALIASES: return ALIASES[key]
    for label in LABELS:
        if key == label.lower(): return label
    return text

def claim_hash(claim: str) -> str:
    return hashlib.sha256(claim.lower().encode("utf-8")).hexdigest()[:20]

def load_dataset() -> tuple[pd.DataFrame, Dict[str, Any]]:
    sci_raw = pd.read_csv(DATA_DIR / "Scientific Dataset.csv")
    pol_raw = pd.read_csv(DATA_DIR / "Political Dataset.csv")
    sci = sci_raw.rename(columns={"Claim":"claim", "Gold_Verdict":"gold_verdict", "Pred_Verdict":"baseline_pred_verdict", "Original label":"original_label"})
    pol = pol_raw.rename(columns={"Claim":"claim", "Gold_Verdict":"gold_verdict", "Pred_Verdict":"baseline_pred_verdict", "Original_Verdict":"original_label"})
    sci["dataset"], pol["dataset"] = "scientific", "political"
    df = pd.concat([sci, pol], ignore_index=True)
    df["claim"] = df["claim"].map(clean_claim)
    df["claim_key"] = df["claim"].map(claim_hash)
    df["gold_verdict"] = df["gold_verdict"].map(normalize_label)
    df["baseline_pred_verdict"] = df["baseline_pred_verdict"].map(normalize_label)
    blank = int(df.claim.eq("").sum())
    invalid = int((~df.gold_verdict.isin(LABELS)).sum())
    duplicate = int(df[df.claim.ne("") & df.gold_verdict.isin(LABELS)].duplicated("claim_key").sum())
    valid = df[df.claim.ne("") & df.gold_verdict.isin(LABELS)].drop_duplicates("claim_key").reset_index(drop=True)
    manifest = {
        "seed": SEED,
        "raw_rows": {"political": int(len(pol_raw)), "scientific": int(len(sci_raw)), "total": int(len(df))},
        "excluded_blank_claims": blank,
        "excluded_non_six_class_labels": invalid,
        "excluded_duplicate_claims": duplicate,
        "valid_unique_claims": int(len(valid)),
        "valid_by_dataset": valid.dataset.value_counts().to_dict(),
        "gold_distribution": {ds: g.gold_verdict.value_counts().to_dict() for ds,g in valid.groupby("dataset")},
        "baseline_name": "inherited_prior_study_baseline",
    }
    return valid, manifest

data, manifest = load_dataset()
display(data.groupby(["dataset", "gold_verdict"]).size().unstack(fill_value=0))
print(json.dumps(manifest, indent=2))


In [ ]:
# Three disjoint splits. Rare classes receive at least one prompt example when possible,
# and calibration receives only records remaining after prompt-development extraction.
rng = np.random.default_rng(SEED)
prompt_parts, calibration_parts, evaluation_parts = [], [], []

for (dataset, label), group in data.groupby(["dataset", "gold_verdict"]):
    order = list(rng.permutation(group.index))
    n_prompt = min(PROMPT_EXAMPLES_PER_CLASS, max(1, len(order)//10))
    remaining_after_prompt = max(0, len(order) - n_prompt)
    n_cal = min(CALIBRATION_PER_CLASS, max(0, remaining_after_prompt//5))
    prompt_parts.append(data.loc[order[:n_prompt]])
    calibration_parts.append(data.loc[order[n_prompt:n_prompt+n_cal]])
    evaluation_parts.append(data.loc[order[n_prompt+n_cal:]])

prompt_development_df = pd.concat(prompt_parts).reset_index(drop=True)
calibration_df = pd.concat(calibration_parts).reset_index(drop=True)
evaluation_pool = pd.concat(evaluation_parts).reset_index(drop=True)

if PILOT_PER_DATASET is None:
    evaluation_df = evaluation_pool.copy()
else:
    n_pilot = min(PILOT_PER_DATASET, int(evaluation_pool.groupby("dataset").size().min()))
    evaluation_df = evaluation_pool.groupby("dataset", group_keys=False).sample(n=n_pilot, random_state=SEED).reset_index(drop=True)

sets = [set(x.claim_key) for x in [prompt_development_df, calibration_df, evaluation_df]]
assert sets[0].isdisjoint(sets[1]) and sets[0].isdisjoint(sets[2]) and sets[1].isdisjoint(sets[2])

manifest.update({
    "prompt_development_claims": int(len(prompt_development_df)),
    "calibration_claims": int(len(calibration_df)),
    "evaluation_pool_claims": int(len(evaluation_pool)),
    "evaluation_run_claims": int(len(evaluation_df)),
    "pilot_per_dataset": PILOT_PER_DATASET,
    "prompt_examples_per_class": PROMPT_EXAMPLES_PER_CLASS,
    "calibration_per_class_cap": CALIBRATION_PER_CLASS,
})
MANIFEST_JSON.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print("Prompt-development:", len(prompt_development_df))
print("Calibration:", len(calibration_df))
print("Evaluation run:", len(evaluation_df))
display(evaluation_df.groupby(["dataset", "gold_verdict"]).size().unstack(fill_value=0))


In [ ]:
SYSTEM_PROMPT = """You are a careful fact-checker performing a six-class PolitiFact-style benchmark.
No web search, browsing, retrieval tool, gold label, prior prediction, or original label is available for the target claim.
You MUST choose exactly one label: Pants on Fire, False, Mostly False, Half True, Mostly True, True.
Never output Unverifiable, Unknown, Not Checkable, or Insufficient Information. Missing context lowers confidence and
sets evidence_gap=true, but does not prevent a best calibrated six-class judgment.

Apply the reasoning gates in this logical order:
1. Pauli-Frankfurt: coherence, falsifiability, truth orientation, scope, and claim type.
2. CRAAP: Currency, Relevance, Authority, Accuracy, and Purpose, each scored 1-5.
3. Ethics: omission, framing, manipulation, SPJ risk, and Rawlsian fairness.
4. Verdict composition: factual correctness controls the truth label. Ethical rhetoric may identify material omission,
but rhetoric alone must not turn a factually supported claim into False.

Label rubric:
True = accurate without important qualification.
Mostly True = accurate core with a minor qualification.
Half True = important accurate and inaccurate/omitted components are balanced.
Mostly False = a small element of truth inside a materially misleading claim.
False = factually wrong.
Pants on Fire = factually wrong and absurd, fabricated, or egregiously misleading; use sparingly."""

def prompt_examples(dataset: str) -> str:
    pool = prompt_development_df[prompt_development_df.dataset.eq(dataset)]
    lines=[]
    for label in LABELS:
        for x in pool[pool.gold_verdict.eq(label)].sort_values("claim_key").itertuples():
            lines.append(f"- Claim: {x.claim}\n  Label: {x.gold_verdict}")
    return "\n".join(lines)

def classification_prompt(claim: str, dataset: str) -> str:
    return f"""{SYSTEM_PROMPT}

Domain: {dataset}
Target claim: {claim}

Prompt-development examples (label-boundary guidance only; never evidence for the target):
{prompt_examples(dataset)}

Return ONLY one valid JSON object:
{{
  "final_verdict": "one allowed label",
  "probabilities": {{"Pants on Fire":0.0,"False":0.0,"Mostly False":0.0,"Half True":0.0,"Mostly True":0.0,"True":0.0}},
  "confidence": 0.0,
  "evidence_gap": false,
  "needs_human_review": false,
  "pauli_frankfurt": {{"coherent":true,"falsifiable":true,"truth_oriented":true,"scope_score":1,"claim_type":"factual","reason":"..."}},
  "craap": {{"currency":1,"relevance":1,"authority":1,"accuracy":1,"purpose":1,"reason":"..."}},
  "ethics": {{"omission":"Low","framing":"Low","manipulation":"Low","spj_risk":"Low","rawls_check":"Pass","reason":"..."}},
  "reasoning_summary": "concise auditable explanation"
}}
All probabilities must be nonnegative and sum to 1. final_verdict must be their argmax."""

def parse_json(text: str) -> Dict[str, Any]:
    cleaned = re.sub(r"^```(?:json)?|```$", "", (text or "").strip(), flags=re.I).strip()
    try: return json.loads(cleaned)
    except Exception:
        match = re.search(r"\{.*\}", cleaned, flags=re.S)
        if match: return json.loads(match.group(0))
        raise ValueError("No valid JSON object")

def safe_float(value: Any, default=np.nan) -> float:
    try: return float(value)
    except Exception: return default

def validate_output(raw: Dict[str, Any]) -> Dict[str, Any]:
    probs = raw.get("probabilities") or {}
    clean = {label:max(0.0, safe_float(probs.get(label), 0.0)) for label in LABELS}
    total = sum(clean.values())
    if total <= 0: raise ValueError("Missing probability distribution")
    clean = {k:v/total for k,v in clean.items()}
    verdict = max(clean, key=clean.get)
    raw["probabilities"], raw["final_verdict"], raw["confidence"] = clean, verdict, float(clean[verdict])
    for required in ["pauli_frankfurt", "craap", "ethics", "reasoning_summary"]:
        if required not in raw: raise ValueError(f"Missing required field: {required}")
    return raw


In [ ]:
@dataclass
class Runner:
    key: str
    model: str
    client: Any

    def complete(self, prompt: str) -> Dict[str, Any]:
        if self.key == "openai":
            r = self.client.chat.completions.create(model=self.model, messages=[{"role":"user","content":prompt}], response_format={"type":"json_object"})
            return validate_output(parse_json(r.choices[0].message.content))
        if self.key == "claude":
            r = self.client.messages.create(model=self.model, max_tokens=1800, messages=[{"role":"user","content":prompt}])
            text = "".join(b.text for b in r.content if getattr(b, "type", "") == "text")
            return validate_output(parse_json(text))
        if self.key == "gemini":
            from google.genai import types
            config = types.GenerateContentConfig(response_mime_type="application/json")
            r = self.client.models.generate_content(model=self.model, contents=prompt, config=config)
            return validate_output(parse_json(r.text))
        raise ValueError(self.key)

    def run(self, claim: str, dataset: str) -> Dict[str, Any]:
        for attempt in range(MAX_RETRIES):
            try:
                out = self.complete(classification_prompt(claim, dataset))
                return {"model_key":self.key,"model_version":self.model,"claim":claim,"dataset":dataset,**out}
            except Exception as exc:
                if attempt + 1 == MAX_RETRIES:
                    return {"model_key":self.key,"model_version":self.model,"claim":claim,"dataset":dataset,
                            "final_verdict":"ERROR","error":repr(exc),"needs_human_review":True}
                time.sleep(RETRY_BASE_SECONDS*(2**attempt)+random.random())

def build_runners() -> Dict[str, Runner]:
    missing=[k for k in ["OPENAI_API_KEY","ANTHROPIC_API_KEY","GOOGLE_API_KEY"] if not os.getenv(k)]
    if missing: raise RuntimeError("Missing API keys: "+", ".join(missing))
    import openai, anthropic
    from google import genai
    return {
        "openai":Runner("openai",OPENAI_MODEL,openai.OpenAI(api_key=os.environ["OPENAI_API_KEY"])),
        "claude":Runner("claude",CLAUDE_MODEL,anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])),
        "gemini":Runner("gemini",GEMINI_MODEL,genai.Client(api_key=os.environ["GOOGLE_API_KEY"])),
    }

def completed_pairs(path: Path) -> set[tuple[str,str]]:
    if not path.exists(): return set()
    pairs=set()
    for line in path.read_text(encoding="utf-8").splitlines():
        if not line.strip(): continue
        row=json.loads(line)
        if row.get("final_verdict") in LABELS:
            pairs.add((row.get("claim_key"),row.get("model_key")))
    return pairs


In [ ]:
run_df = pd.concat([
    calibration_df.assign(split="calibration"),
    evaluation_df.assign(split="evaluation"),
], ignore_index=True)

if RUN_LIVE_APIS:
    runners=build_runners()
    done=completed_pairs(RESULTS_JSONL)
    dataset_totals=run_df.groupby("dataset").size().to_dict()
    dataset_seen=Counter(); overall_total=len(run_df)
    with RESULTS_JSONL.open("a",encoding="utf-8") as f:
        for overall_i,row in enumerate(run_df.itertuples(index=False),start=1):
            dataset_seen[row.dataset]+=1
            print(f"\n[{row.dataset.title()} claim {dataset_seen[row.dataset]}/{dataset_totals[row.dataset]} | overall {overall_i}/{overall_total}]")
            print(row.claim[:180]+("..." if len(row.claim)>180 else ""))
            pending={k:r for k,r in runners.items() if (row.claim_key,k) not in done}
            skipped=sorted(set(runners)-set(pending))
            if skipped: print("Already saved:",", ".join(skipped))
            if not pending:
                print("Status: complete (resumed; no API calls)"); continue
            with ThreadPoolExecutor(max_workers=min(MAX_PARALLEL_PROVIDERS,len(pending))) as pool:
                futures={pool.submit(r.run,row.claim,row.dataset):k for k,r in pending.items()}
                for future in as_completed(futures):
                    model_key=futures[future]; result=future.result()
                    result.update({"claim_key":row.claim_key,"split":row.split})
                    f.write(json.dumps(result,ensure_ascii=False)+"\n"); f.flush()
                    if result.get("final_verdict") in LABELS: done.add((row.claim_key,model_key))
                    print(f"  {model_key}: {result.get('final_verdict')}")
            print(f"Status: {len(done)} valid model/claim pairs saved")
else:
    print("Dry run only. Set RUN_LIVE_APIS=True to execute.")


In [ ]:
def load_results() -> pd.DataFrame:
    if not RESULTS_JSONL.exists(): return pd.DataFrame()
    rows=[json.loads(x) for x in RESULTS_JSONL.read_text(encoding="utf-8").splitlines() if x.strip()]
    return pd.DataFrame(rows).drop_duplicates(["claim_key","model_key"],keep="last")

results=load_results()
if results.empty:
    print("No results yet.")
else:
    print("Model rows:",len(results))
    print(pd.crosstab(results.model_key,results.final_verdict,margins=True))
    print("ERROR rows:",int(results.final_verdict.eq("ERROR").sum()))


In [ ]:
def prob_vector(value: Any) -> np.ndarray:
    if isinstance(value,str): value=json.loads(value)
    return np.array([safe_float((value or {}).get(label),0.0) for label in LABELS],dtype=float)

def craap_values(value: Any) -> Dict[str,float]:
    if isinstance(value,str): value=json.loads(value)
    return {k:safe_float((value or {}).get(k),1.0) for k in ["currency","relevance","authority","accuracy","purpose"]}

def ethics_values(value: Any) -> Dict[str,str]:
    if isinstance(value,str): value=json.loads(value)
    return {k:str((value or {}).get(k,"Low")) for k in ["omission","framing","manipulation","spj_risk","rawls_check"]}

def risk_score(value: Any) -> float:
    e=ethics_values(value)
    return max(RISK_TO_SCORE.get(e[k].lower(),0.0) for k in ["omission","framing","manipulation","spj_risk"])

def learn_reliability(cal: pd.DataFrame) -> Dict[str,Dict[str,float]]:
    merged=cal.merge(calibration_df[["claim_key","dataset","gold_verdict"]],on=["claim_key","dataset"])
    out={}
    for dataset,g in merged.groupby("dataset"):
        out[dataset]={m:float(x.final_verdict.eq(x.gold_verdict).mean()) for m,x in g.groupby("model_key")}
    out["all"]={m:float(x.final_verdict.eq(x.gold_verdict).mean()) for m,x in merged.groupby("model_key")}
    return out

def reliability_of(rel,dataset,model):
    return float(rel.get(dataset,{}).get(model,rel.get("all",{}).get(model,0.05)))

def aggregate_gate_outputs(g: pd.DataFrame) -> Dict[str,Any]:
    craap_rows=[craap_values(x) for x in g.craap]
    aggregated={k:float(np.mean([x[k] for x in craap_rows])) for k in craap_rows[0]}
    ethics_rows=[ethics_values(x) for x in g.ethics]
    risks=[risk_score(x) for x in g.ethics]
    worst=int(np.argmax(risks))
    return {
        "aggregated_craap":float(np.mean(list(aggregated.values()))),
        **{f"craap_{k}":v for k,v in aggregated.items()},
        "consolidated_ethical_risk":["Low","Medium","High"][int(round(max(risks)*2))],
        "consolidated_ethics":json.dumps(ethics_rows[worst],ensure_ascii=False),
    }

def majority_voting(g,rel,dataset):
    counts=Counter(g.final_verdict); top=max(counts.values()); candidates=[x for x,n in counts.items() if n==top]
    mean_probs=np.mean(np.stack(g.probabilities.map(prob_vector)),axis=0)
    winner=max(candidates,key=lambda x:mean_probs[LABELS.index(x)])
    return {"strategy":"majority_voting","ensemble_verdict":winner,"ensemble_confidence":top/len(g),
            "needs_human_review":top/len(g)<2/3,"meta_reasoning":f"{top}/{len(g)} experts voted {winner}."}

def confidence_weighted_voting(g,rel,dataset):
    vectors=np.stack(g.probabilities.map(prob_vector)); weights=np.array([max(safe_float(x),0.01) for x in g.confidence])
    probs=np.average(vectors,axis=0,weights=weights); winner=LABELS[int(np.argmax(probs))]
    return {"strategy":"confidence_weighted_voting","ensemble_verdict":winner,"ensemble_confidence":float(probs.max()),
            "needs_human_review":float(probs.max())<.45,"meta_reasoning":"Expert probability vectors weighted by claim-level confidence.",
            **{f"p_{LABELS[i]}":float(probs[i]) for i in range(len(LABELS))}}

def calibration_accuracy_weighting(g,rel,dataset):
    vectors=np.stack(g.probabilities.map(prob_vector)); weights=np.array([max(reliability_of(rel,dataset,m),.05)**2 for m in g.model_key])
    probs=np.average(vectors,axis=0,weights=weights); winner=LABELS[int(np.argmax(probs))]
    return {"strategy":"calibration_accuracy_weighting","ensemble_verdict":winner,"ensemble_confidence":float(probs.max()),
            "needs_human_review":float(probs.max())<.45,"meta_reasoning":"Expert probabilities weighted by disjoint calibration accuracy.",
            **{f"p_{LABELS[i]}":float(probs[i]) for i in range(len(LABELS))}}

def gate_specific_routing(g,rel,dataset):
    work=g.copy(); work["reliability"]=[reliability_of(rel,dataset,m) for m in work.model_key]
    work["craap_mean"]=[np.mean(list(craap_values(x).values()))/5 for x in work.craap]
    work["risk"]=[risk_score(x) for x in work.ethics]
    work["route_score"]=.55*work.reliability+.30*work.confidence+.15*work.craap_mean
    factual=work.sort_values(["route_score","confidence"],ascending=False).iloc[0]
    epistemic=work.sort_values(["craap_mean","reliability"],ascending=False).iloc[0]
    ethical=work.sort_values(["risk","reliability"],ascending=False).iloc[0]
    agreement=max(Counter(work.final_verdict).values())/len(work)
    return {"strategy":"gate_specific_routing","ensemble_verdict":factual.final_verdict,"ensemble_confidence":float(factual.confidence),
            "needs_human_review":bool(agreement<2/3 or factual.confidence<.45 or ethical.risk>=1),
            "factual_expert":factual.model_key,"epistemic_expert":epistemic.model_key,"ethics_expert":ethical.model_key,
            "meta_reasoning":f"Factual={factual.model_key}; CRAAP={epistemic.model_key}; Ethics={ethical.model_key}."}

def build_ensembles(evaluation_results,rel):
    rows=[]; valid=evaluation_results[evaluation_results.final_verdict.isin(LABELS)]
    for claim_key,g in valid.groupby("claim_key"):
        dataset=str(g.dataset.iloc[0]); counts=Counter(g.final_verdict); agreement=max(counts.values())/len(g)
        shared={"claim_key":claim_key,"dataset":dataset,"valid_model_count":len(g),"agreement":agreement,
                "verdict_distribution":json.dumps(counts,ensure_ascii=False),**aggregate_gate_outputs(g)}
        for fn in [majority_voting,confidence_weighted_voting,gate_specific_routing,calibration_accuracy_weighting]:
            rows.append({**shared,**fn(g,rel,dataset)})
    return pd.DataFrame(rows)

if not results.empty:
    calibration_results=results[results.split.eq("calibration") & results.final_verdict.isin(LABELS)]
    reliability=learn_reliability(calibration_results)
    print("Calibration reliability:",json.dumps(reliability,indent=2))
    ensemble_df=build_ensembles(results[results.split.eq("evaluation")],reliability)
    ensemble_df=ensemble_df.merge(evaluation_df[["claim_key","dataset","claim","gold_verdict","baseline_pred_verdict"]],on=["claim_key","dataset"])
    ensemble_df.to_csv(ENSEMBLE_CSV,index=False)
    display(ensemble_df.head())


In [ ]:
def confusion_matrix(y_true,y_pred,labels=LABELS):
    cm=np.zeros((len(labels),len(labels)),dtype=int); pos={x:i for i,x in enumerate(labels)}
    for a,b in zip(y_true,y_pred):
        if a in pos and b in pos: cm[pos[a],pos[b]]+=1
    return cm

def expected_calibration_error(y_true,y_pred,confidence,bins=10):
    y_true=np.asarray(y_true); y_pred=np.asarray(y_pred); confidence=np.clip(np.asarray(confidence,dtype=float),0,1)
    correct=(y_true==y_pred).astype(float); total=len(correct)
    if not total:return np.nan
    ece=0.0
    for lo,hi in zip(np.linspace(0,1,bins+1)[:-1],np.linspace(0,1,bins+1)[1:]):
        mask=(confidence>=lo)&(confidence<hi if hi<1 else confidence<=hi)
        if mask.any(): ece+=mask.mean()*abs(correct[mask].mean()-confidence[mask].mean())
    return float(ece)

def multiclass_brier(y_true,probabilities):
    if not len(y_true):return np.nan
    target=np.zeros((len(y_true),len(LABELS)))
    for i,label in enumerate(y_true): target[i,LABELS.index(label)]=1
    return float(np.mean(np.sum((np.stack(probabilities)-target)**2,axis=1)))

def classification_metrics(df,pred_col,conf_col=None,prob_col=None):
    temp=df[df.gold_verdict.isin(LABELS)&df[pred_col].isin(LABELS)].copy(); y=temp.gold_verdict.tolist(); p=temp[pred_col].tolist(); n=len(temp)
    if not n:return {"n":0}
    cm=confusion_matrix(y,p); per=[]
    for i in range(len(LABELS)):
        tp=cm[i,i]; fp=cm[:,i].sum()-tp; fn=cm[i,:].sum()-tp
        pr=tp/(tp+fp) if tp+fp else 0; rc=tp/(tp+fn) if tp+fn else 0; per.append((pr,rc,2*pr*rc/(pr+rc) if pr+rc else 0))
    acc=float(np.trace(cm)/n); row=cm.sum(axis=1); col=cm.sum(axis=0); pe=float(np.sum(row*col)/(n*n)); kappa=(acc-pe)/(1-pe) if pe!=1 else np.nan
    out={"n":n,"accuracy":acc,"macro_precision":float(np.mean([x[0] for x in per])),"macro_recall":float(np.mean([x[1] for x in per])),
         "macro_f1":float(np.mean([x[2] for x in per])),"cohen_kappa":float(kappa),"ece":np.nan,"brier_score":np.nan}
    if conf_col and conf_col in temp: out["ece"]=expected_calibration_error(y,p,temp[conf_col].fillna(.5).tolist())
    if prob_col and prob_col in temp: out["brier_score"]=multiclass_brier(y,[prob_vector(x) for x in temp[prob_col]])
    return out

def ensemble_probabilities(row):
    vals=[row.get(f"p_{label}") for label in LABELS]
    if all(pd.notna(x) for x in vals): return {label:float(vals[i]) for i,label in enumerate(LABELS)}
    return {label:1.0 if label==row.ensemble_verdict else 0.0 for label in LABELS}

if not results.empty:
    labeled_models=results[results.split.eq("evaluation")].merge(evaluation_df[["claim_key","gold_verdict"]],on="claim_key")
    metric_rows=[]
    for (model,dataset),g in labeled_models.groupby(["model_key","dataset"]): metric_rows.append({"system":f"model:{model}","dataset":dataset,**classification_metrics(g,"final_verdict","confidence","probabilities")})
    for model,g in labeled_models.groupby("model_key"): metric_rows.append({"system":f"model:{model}","dataset":"all",**classification_metrics(g,"final_verdict","confidence","probabilities")})
    ensemble_df["ensemble_probabilities"]=[ensemble_probabilities(x) for _,x in ensemble_df.iterrows()]
    for (strategy,dataset),g in ensemble_df.groupby(["strategy","dataset"]): metric_rows.append({"system":f"ensemble:{strategy}","dataset":dataset,**classification_metrics(g,"ensemble_verdict","ensemble_confidence","ensemble_probabilities")})
    for strategy,g in ensemble_df.groupby("strategy"): metric_rows.append({"system":f"ensemble:{strategy}","dataset":"all",**classification_metrics(g,"ensemble_verdict","ensemble_confidence","ensemble_probabilities")})
    baseline=evaluation_df[evaluation_df.claim_key.isin(ensemble_df.claim_key.unique())]
    for dataset,g in baseline.groupby("dataset"): metric_rows.append({"system":"inherited_prior_study_baseline","dataset":dataset,**classification_metrics(g.rename(columns={"baseline_pred_verdict":"prediction"}),"prediction")})
    metric_rows.append({"system":"inherited_prior_study_baseline","dataset":"all",**classification_metrics(baseline.rename(columns={"baseline_pred_verdict":"prediction"}),"prediction")})
    metrics_df=pd.DataFrame(metric_rows); metrics_df.to_csv(METRICS_CSV,index=False); display(metrics_df)


In [ ]:
def exact_mcnemar_p(b,c):
    n=b+c
    if n==0:return 1.0
    tail=sum(math.comb(n,k) for k in range(min(b,c)+1))/(2**n)
    return min(1.0,2*tail)

def holm_adjust(pvalues):
    order=np.argsort(pvalues); adjusted=np.empty(len(pvalues)); running=0.0; m=len(pvalues)
    for rank,idx in enumerate(order):
        value=min(1.0,(m-rank)*pvalues[idx]); running=max(running,value); adjusted[idx]=running
    return adjusted

def wide_predictions():
    base=evaluation_df[["claim_key","dataset","gold_verdict","baseline_pred_verdict"]].copy()
    model_wide=results[results.split.eq("evaluation")].pivot(index="claim_key",columns="model_key",values="final_verdict").add_prefix("model_").reset_index()
    ens_wide=ensemble_df.pivot(index="claim_key",columns="strategy",values="ensemble_verdict").add_prefix("ensemble_").reset_index()
    return base.merge(model_wide,on="claim_key",how="left").merge(ens_wide,on="claim_key",how="left")

if not results.empty:
    wide=wide_predictions(); systems=[c for c in wide if c.startswith("ensemble_")]
    comparisons=[]
    reference_cols=["baseline_pred_verdict","model_openai","model_claude","model_gemini"]
    for dataset,g in [("all",wide),*list(wide.groupby("dataset"))]:
        for ens in systems:
            for ref in reference_cols:
                temp=g.dropna(subset=[ens,ref,"gold_verdict"]); a=temp[ref].eq(temp.gold_verdict); b=temp[ens].eq(temp.gold_verdict)
                awb=int((a&~b).sum()); bwa=int((~a&b).sum())
                comparisons.append({"dataset":dataset,"system_a":ref,"system_b":ens,"n":len(temp),"a_correct_b_wrong":awb,"a_wrong_b_correct":bwa,"mcnemar_exact_p":exact_mcnemar_p(awb,bwa)})
    mc=pd.DataFrame(comparisons); mc["holm_adjusted_p"]=holm_adjust(mc.mcnemar_exact_p.to_numpy()); mc["significant_0_05"]=mc.holm_adjusted_p<.05; mc.to_csv(MCNEMAR_CSV,index=False)

    dis_rows=[]
    for claim_key,g in results[results.split.eq("evaluation") & results.final_verdict.isin(LABELS)].groupby("claim_key"):
        labels_out=g.final_verdict.tolist(); scores=[LABEL_TO_SCORE[x] for x in labels_out]; counts=Counter(labels_out)
        dis_rows.append({"claim_key":claim_key,"dataset":g.dataset.iloc[0],"model_count":len(g),"unique_verdict_count":len(counts),
            "unanimous":len(counts)==1,"agreement":max(counts.values())/len(g),"max_label_distance":max(scores)-min(scores),
            "mean_confidence":float(g.confidence.mean()),"model_human_review_flags":int(g.needs_human_review.fillna(False).sum()),
            "verdict_distribution":json.dumps(counts,ensure_ascii=False),"openai_verdict":g.set_index("model_key").final_verdict.get("openai"),
            "claude_verdict":g.set_index("model_key").final_verdict.get("claude"),"gemini_verdict":g.set_index("model_key").final_verdict.get("gemini")})
    disagreement=pd.DataFrame(dis_rows).merge(evaluation_df[["claim_key","claim","gold_verdict"]],on="claim_key"); disagreement.to_csv(DISAGREEMENT_CSV,index=False)
    display(mc.head()); display(disagreement.sort_values(["max_label_distance","unique_verdict_count"],ascending=False).head(20))


In [ ]:
if not results.empty:
    eval_results=results[results.split.eq("evaluation")]
    complete_claims=int((eval_results.groupby("claim_key").model_key.nunique()==3).sum())
    summary={
        "models":{"openai":OPENAI_MODEL,"claude":CLAUDE_MODEL,"gemini":GEMINI_MODEL},
        "search_enabled":False,
        "prompt_development_claims":len(prompt_development_df),"calibration_claims":len(calibration_df),"evaluation_target_claims":len(evaluation_df),
        "target_model_rows":len(evaluation_df)*3,"saved_evaluation_model_rows":len(eval_results),"complete_three_model_claims":complete_claims,
        "error_rows":int(results.final_verdict.eq("ERROR").sum()),"abstention_rows":int(results.final_verdict.isin(["Unverifiable","Not Checkable"]).sum()),
        "strategies":["majority_voting","confidence_weighted_voting","gate_specific_routing","calibration_accuracy_weighting"],
        "output_files":[str(x) for x in [RESULTS_JSONL,ENSEMBLE_CSV,METRICS_CSV,MCNEMAR_CSV,DISAGREEMENT_CSV,MANIFEST_JSON]],
    }
    SUMMARY_JSON.write_text(json.dumps(summary,indent=2),encoding="utf-8")
    print(json.dumps(summary,indent=2))
    print("\nFINAL RUN GATE")
    print("1. ERROR rows must be 0; failed rows automatically retry on rerun.")
    print("2. Each evaluation claim must have all three models.")
    print("3. Review at least 20 highest-disagreement cases.")
    print("4. Treat the inherited prior-study baseline as non-reproduced and methodologically limited.")


## Final deliverables generated

- `model_outputs.jsonl`: full per-model structured gate outputs and reasoning summaries.
- `ensemble_predictions.csv`: four aggregation strategies with CRAAP, ethics, confidence, and meta-reasoning.
- `benchmark_metrics.csv`: single-model and ensemble Accuracy, Macro Precision/Recall/F1, Cohen's Kappa, ECE, and Brier score.
- `mcnemar_tests.csv`: exact McNemar comparisons with Holm correction.
- `disagreement_analysis.csv`: consensus, label distance, human-review flags, and the 20-case qualitative-review queue.
- `dataset_manifest.json`: source row counts, exclusions, label distributions, and split sizes.
- `run_summary.json`: execution completeness and quality checks.

For the final benchmark, keep `SEED`, prompt-development split, calibration split, model versions, and prompts frozen. Change only `PILOT_PER_DATASET` from a number to `None`; the JSONL resume logic prevents duplicate successful API calls.
